# **Required Dependencies**

In [ ]:
import os
import pandas as pd 
import numpy as np 
import seaborn as sns
import matplotlib.pyplot as plt
import cv2 
import tensorflow as tf
import random
from tqdm import tqdm
from keras.layers import Conv2D, MaxPooling2D , GlobalAveragePooling2D , BatchNormalization ,Dropout ,Flatten , Dense , Input
from tensorflow.keras.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
    ReduceLROnPlateau
)
from keras.models import Sequential
from matplotlib.patches import FancyBboxPatch

import warnings

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    
plt.style.use('ggplot')

# RANDOM FOREST DEPENDENCIES
from sklearn.model_selection import train_test_split
from sklearn.metrics import ( classification_report, confusion_matrix, accuracy_score, precision_recall_fscore_support ) 
from sklearn.ensemble import RandomForestClassifier 
import time 
import joblib


In [3]:
dataDir = '/kaggle/input/plantdisease/PlantVillage'
selectedClasses = [
    'Tomato_Bacterial_spot',
    'Tomato_Early_blight',
    'Tomato_Late_blight',
    'Tomato_Leaf_Mold',
    'Tomato_Septoria_leaf_spot',
    'Tomato_Spider_mites_Two_spotted_spider_mite',
    'Tomato__Target_Spot',
    'Tomato__Tomato_YellowLeaf__Curl_Virus',
    'Tomato__Tomato_mosaic_virus',
    'Tomato_healthy'
]

In [ ]:
imgPaths = []
labels = []

for className in selectedClasses:                       # respect this order
    classPath = os.path.join(dataDir, className)
    if not os.path.isdir(classPath):
        continue                                        # skip if folder missing
    
    # optional: make image order deterministic too
    for img in sorted(os.listdir(classPath)):
        imgPath = os.path.join(classPath, img)
        imgPaths.append(imgPath)
        labels.append(className)

df = pd.DataFrame({
    'label': labels,
    'imgPath': imgPaths
})

df

In [5]:
# Save df as a CSV file in the working directory
output_path = '/kaggle/working/tomato_images_df.csv'
df.to_csv(output_path, index=False)
print("Saved to:", output_path)

Saved to: /kaggle/working/tomato_images_df.csv


In [6]:
def save_figure(fig, class_name, filename=None, base_dir="/kaggle/working"):
    """
    Save a Matplotlib figure to disk, using the class name to build
    a default filename if none is provided.
    """
    if filename is None:
        safe_name = class_name.replace(" ", "_")
        filename = f"{safe_name}_samples.png"
    
    out_path = os.path.join(base_dir, filename)
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print("Saved figure to:", out_path)
    return out_path

In [7]:
def show_samples(class_name, save=False, filename=None):
    """
    Show 4 random sample images for a given tomato disease class
    in a 2x2 grid (with rounded corners) and optionally save the figure.
    """
    if class_name not in selectedClasses:
        print(f"Class '{class_name}' not found in selectedClasses.")
        return
    
    class_rows = df[df['label'] == class_name]
    if class_rows.empty:
        print(f"No images found in df for class '{class_name}'.")
        return
    
    n_samples = min(3, len(class_rows))
    sample_indices = random.sample(list(class_rows.index), n_samples)
    
    fig, axs = plt.subplots(1, 3, figsize=(8, 8))
    axs = axs.flatten()
    
    for i in range(3):
        ax = axs[i]
        if i < n_samples:
            row = df.loc[sample_indices[i]]
            img_path = row['imgPath']
            
            img = cv2.imread(img_path)
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            im = ax.imshow(img)
            ax.axis('off')
            
            # Rounded corners
            radius = 0.1
            round_box = FancyBboxPatch(
                (0, 0), 1, 1,
                boxstyle=f"round,pad=0,rounding_size={radius}",
                transform=ax.transAxes,
                linewidth=0,
                facecolor="none"
            )
            ax.add_patch(round_box)
            im.set_clip_path(round_box)
        else:
            ax.axis('off')

    # use the separate save function
    if save:
        save_figure(fig, class_name, filename)

    plt.show()


In [ ]:
show_samples("Tomato__Tomato_YellowLeaf__Curl_Virus")

# **Image Preprocessing**

**Label Encoding**

In [9]:
def encode_labels(df, selected_classes, column_name="label"): 
    # Create a mapping from class name to numeric label
    class_to_label = {}

    for index, class_name in enumerate(selected_classes):
        class_to_label[class_name] = index

    # Replace class names with numeric labels
    df[column_name] = df[column_name].map(class_to_label)

    return df

In [ ]:
df = encode_labels(df, selectedClasses)
df

**PreProcessing**

In [11]:
def remove_invalid_data(df):
    
    invalidIndexes = []

    for index, imagePath in tqdm(
        enumerate(df["imgPath"]),
        total=len(df),
        desc="Checking images"
    ):

        image = cv2.imread(imagePath)

        if image is None:
            print(f"Cannot read image: {imagePath}")
            invalidIndexes.append(index)

    cleanedDf = df.drop(df.index[invalidIndexes]).reset_index(drop=True)

    print(f"\nRemoved {len(invalidIndexes)} invalid images.")
    print(f"Remaining images: {len(cleanedDf)}")

    return cleanedDf

In [12]:
df = remove_invalid_data(df)

Checking images:  88%|████████▊ | 14063/16012 [03:02<00:29, 65.13it/s] 

Cannot read image: /kaggle/input/plantdisease/PlantVillage/Tomato__Tomato_YellowLeaf__Curl_Virus/svn-r6Yb5c


Checking images: 100%|██████████| 16012/16012 [03:27<00:00, 77.04it/s] 


Removed 1 invalid images.
Remaining images: 16011


In [13]:
def loadAndPreprocessImages(imagePaths, imageSize=(150, 150)):
    images = []

    for imagePath in tqdm(imagePaths, total=len(imagePaths)):
        
        image = cv2.imread(imagePath)

        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        image = cv2.resize(image, imageSize)

        images.append(image)

    return images

In [14]:
imgs = loadAndPreprocessImages(df["imgPath"])

# convert images and labels to a numpy array so we can split them 
images = np.array(imgs)
labels = np.array(df['label'])

len(images) , len(labels)

100%|██████████| 16011/16011 [00:48<00:00, 327.03it/s]


(16011, 16011)

In [15]:
# normalize from 0 --> 255 to 0 --> 1  to reduce the execution time 
images = images / 255.0  

**visualize**

In [ ]:
def plot_with_labels(images, labels, rows=3, cols=4, figsize=(25, 10)):
    
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = axes.flatten()

    totalImages = rows * cols

    for i in range(totalImages):
        randomIndex = random.randint(0, len(images) - 1)

        axes[i].imshow(images[randomIndex])
        axes[i].set_title(labels[randomIndex], fontsize=14)
        axes[i].axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
plot_with_labels(images, labels)

# Data Splitting

In [18]:
SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)

random.seed(SEED)
np.random.seed(SEED)
tf.keras.utils.set_random_seed(SEED)

In [19]:
X_train, X_temp, y_train, y_temp = train_test_split(
    images, labels, test_size=0.3, random_state=SEED, shuffle=True, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=SEED, shuffle=True, stratify=y_temp
)
print(f'Shape of X_train : {X_train.shape}')
print(f'Shape of X_test : {X_test.shape}')
print(f'Shape of y_train : {y_train.shape}')
print(f'Shape of y_test : {y_test.shape}')

Shape of X_train : (11207, 150, 150, 3)
Shape of X_test : (2402, 150, 150, 3)
Shape of y_train : (11207,)
Shape of y_test : (2402,)


# CNN Model

In [20]:
def buildCNNModel(
    numClasses,
    inputShape=(150, 150, 3),
    convFilters=[32, 32, 32],
    kernelSize=(3, 3),
    poolSize=(2, 2),
    strides=(2,2),
    activation="relu",
    padding="same",
    dropoutRates=[0.1, 0.1, 0.1],
    denseUnits=128,
):
    """
    ساخت مدل CNN به صورت داینامیک

    Parameters
    ----------
    inputShape : tuple
        ابعاد تصویر ورودی.

    numClasses : int
        تعداد کلاس‌های خروجی.

    convFilters : list
        تعداد فیلتر هر لایه کانولوشن.

    kernelSize : tuple
        اندازه کرنل.

    poolSize : tuple
        اندازه MaxPooling.

    activation : str
        تابع فعال‌سازی.

    padding : str
        نوع Padding.

    dropoutRates : list
        نرخ Dropout برای هر بلوک.

    denseUnits : int
        تعداد نورون‌های Dense.
    """

    model = Sequential()
    
    model.add(Input(shape=inputShape))

    for index, filters in enumerate(convFilters):

        model.add(
        Conv2D(
            filters=filters,
            kernel_size=kernelSize,
            activation=activation,
            padding=padding
        )
    )

        model.add(BatchNormalization())

        model.add(MaxPooling2D(pool_size=poolSize,strides=strides))

        if index < len(dropoutRates):
            model.add(Dropout(dropoutRates[index]))

    model.add(
        GlobalAveragePooling2D(
            name="feature_vector"
        )
    )

    model.add(Dense(denseUnits, activation=activation, name="deep_features"))

    model.add(Dense(numClasses, activation="softmax"))

    return model

In [21]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


**Build Model**

In [22]:
numClasses = len(np.unique(labels))
model = buildCNNModel(numClasses=numClasses)

I0000 00:00:1788704772.584943     138 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 15511 MB memory:  -> device: 0, name: Tesla P100-PCIE-16GB, pci bus id: 0000:00:04.0, compute capability: 6.0


### **Compile**

In [23]:
optimizer = tf.keras.optimizers.AdamW(
    # learning_rate=0.0001,
    # weight_decay=0.0001,
    learning_rate=1e-4,
    weight_decay=1e-5,
    clipnorm=1.0
)
# model.compile(optimizer=optimizer,loss='sparse_categorical_crossentropy' ,metrics=['accuracy'])
model.compile(optimizer='adam',loss='sparse_categorical_crossentropy' ,metrics=['accuracy'])

### **Callbacks**

In [24]:
callbacks = [
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    ),
    EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    ModelCheckpoint(
        filepath="/kaggle/working/best_cnn_model.keras",
        monitor="val_loss",
        save_best_only=True,
        verbose=1
    )
]

In [ ]:
history = model.fit(X_train, y_train,
                     validation_data=(X_val, y_val),
                     epochs=50,
                     batch_size=50,
                     callbacks=callbacks)

In [ ]:
y_pred = model.predict(X_test)

In [ ]:
def plot_history(
    history,
    title,
    ylabel,
    train_param,
    validation_param,
    xlabel="epoch"
):

    plt.plot(history.history[train_param], label="Train")
    plt.plot(history.history[validation_param], label="Validation")

    plt.title(title)
    plt.ylabel(ylabel)
    plt.xlabel(xlabel)
    plt.legend(['train', 'validation'], loc='upper left')
    plt.show()

In [ ]:
plot_history(history=history,title="Model Accuracy",ylabel="accuracy",train_param="accuracy",validation_param="val_accuracy")

In [ ]:
plot_history(history=history,title="Model Loss",ylabel="loss",train_param="loss",validation_param="val_loss")

In [ ]:
best_model = tf.keras.models.load_model(
    "best_cnn_model.keras"
)

test_loss, test_accuracy = best_model.evaluate(
    X_test,
    y_test,
    verbose=1
)

print("Test accuracy:", test_accuracy)
print("Test loss:", test_loss)

In [ ]:
y_test[:10]

In [ ]:
new_y_pred = [np.argmax(x) for x in y_pred]
new_y_pred[:10]

In [ ]:
CM = confusion_matrix(y_test, new_y_pred, labels=np.arange(len(selectedClasses)))
CM_percent = CM.astype('float') / CM.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(12, 9))
sns.heatmap(
    CM_percent,
    annot=True,
    fmt='.2f',
    cmap='summer',
    xticklabels=selectedClasses,
    yticklabels=selectedClasses,
    vmin=0, vmax=100
)
plt.title("CNN–Softmax Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
ClassificationReport = classification_report(y_test,new_y_pred)
print('Classification Report is : \n', ClassificationReport )

# **Random Forest Classification**

In [ ]:
cnn_model = tf.keras.models.load_model("/kaggle/input/models/alitafreshi/final-cnn-model/keras/default/1/95_cnn_model.keras")

### **Feature Extractor**

In [ ]:
feature_extractor = tf.keras.Model(inputs=cnn_model.inputs[0], outputs=cnn_model.get_layer("deep_features").output)

### **Feature Extraction**

In [ ]:
X_train_features = feature_extractor.predict(X_train, batch_size=64, verbose=1)
X_test_features = feature_extractor.predict(X_test, batch_size=64, verbose=1)
print("Train feature shape:", X_train_features.shape)
print("Test feature shape:", X_test_features.shape)

### **Random Forest Classifier**

In [ ]:
random_forest = RandomForestClassifier( 
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    class_weight="balanced_subsample",
    random_state=SEED, n_jobs=-1
)

In [ ]:
rf_train_start = time.perf_counter()
random_forest.fit(X_train_features, y_train)
rf_train_time = time.perf_counter() - rf_train_start

rf_predict_start = time.perf_counter()
rf_predictions = random_forest.predict(X_test_features)
rf_predict_time = time.perf_counter() - rf_predict_start

print(f"Random Forest train time: {rf_train_time:.2f} seconds")
print(f"Random Forest prediction time: {rf_predict_time:.2f} seconds")

In [ ]:
rf_accuracy = accuracy_score(y_test, rf_predictions)

print(f"Random Forest test accuracy: {rf_accuracy:.4f}")

In [ ]:
rf_classification_report =  classification_report(
        y_test,
        rf_predictions,
        labels=np.arange(len(selectedClasses)),
        target_names=selectedClasses,
        output_dict=True,
        zero_division=0
    )
print('Random Forest Classification Report is : \n', rf_classification_report )

In [ ]:
def save_classification_report(report, file_name="classification_report.xlsx"):
    """
    ذخیره Classification Report در فایل اکسل

    report باید با output_dict=True ساخته شده باشد.
    """

    report_df = pd.DataFrame(report).transpose()
    report_df.index.name = "Class"

    report_df.to_excel(file_name)

    print(f"فایل ذخیره شد: {file_name}")

In [ ]:
save_classification_report(
    rf_classification_report,
    "rf_classification_report.xlsx"
)

In [ ]:
rf_cm = confusion_matrix(
    y_test,
    rf_predictions,
    labels=np.arange(len(selectedClasses))
)

rf_cm_percent = rf_cm.astype('float') / rf_cm.sum(axis=1, keepdims=True) * 100


plt.figure(figsize=(12, 9))
sns.heatmap(
    rf_cm_percent,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=selectedClasses,
    yticklabels=selectedClasses
)
plt.title("CNN Features + Random Forest Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.xticks(rotation=90)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()